In [1]:
# ==============================================================================
# 🗺️ CODE MODEL BINER BALANCED DATA (MULTI-ALGORITMA)
# Dataset: unik_per_bulan.csv (Balancing Dinamis Proporsional)
# Upgrade: Wider Hyperparameter Grid Search
# ==============================================================================

import pandas as pd
import numpy as np
import os, gc, time, joblib, warnings
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import (accuracy_score, f1_score, cohen_kappa_score, 
                             precision_score, recall_score, confusion_matrix)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline 

# --- IMPORT SEMUA MODEL ---
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier # CART
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')

# ==========================================================
# 🎯 1. CONTROL PANEL (GANTI MODEL DI SINI)
# ==========================================================
# Pilihan Tersedia: 'Logistic Regression', 'CART', 'Naive Bayes', 'KNN', 
#                   'Random Forest', 'XGBoost', 'LightGBM', 'CatBoost'

MODEL_PILIHAN = 'CART'  # <--- SILAKAN GANTI NAMA MODEL DI SINI

print(f"🔥 MEMULAI NOTEBOOK KHUSUS UNTUK: {MODEL_PILIHAN.upper()} 🔥\n")

# ==========================================================
# 🔍 2. PENCARIAN & PEMUATAN DATASET
# ==========================================================
dataset_path = '/kaggle/input/datasets/jejehayu/data-klasifikasi-padi/unik_per_bulan.csv'
print(f"🔍 Membaca dataset dari: {dataset_path}")

if os.path.exists(dataset_path):
    kolom_input = ['id', 'longitude', 'latitude', 'Tahun', 'Bulan', 'date_img', 'Lahan',
                   'VV', 'VH', 'RVI', 'NDVI', 'LSWI', 'EVI', 'BSI']
    
    df = pd.read_csv(dataset_path, names=kolom_input, header=0)
    df['date_img'] = pd.to_datetime(df['date_img'])

    # Eksklusi data tahun 2021 jika ada
    df = df[df['Tahun'] != 2021].reset_index(drop=True)
    print(f"✅ Dataset berhasil dimuat! Jumlah baris: {len(df)}")
else:
    raise FileNotFoundError(f"❌ ERROR: File {dataset_path} tidak ditemukan.")

# ==========================================================
# ⚙️ 3. FEATURE ENGINEERING (AMAN DARI LEAKAGE)
# ==========================================================
df['segmen'] = df['longitude'].astype(str) + '_' + df['latitude'].astype(str)

# ----------------------------------------------------------
# SORTING KE-1: Spasial Dulu (Untuk Keamanan Kalkulasi Lag)
# ----------------------------------------------------------
df = df.sort_values(['segmen', 'Tahun', 'Bulan']).reset_index(drop=True)

# A. Cyclical Encoding (Musim)
df['Bulan_sin'] = np.sin(2 * np.pi * df['Bulan'] / 12)
df['Bulan_cos'] = np.cos(2 * np.pi * df['Bulan'] / 12)

# B. Satellite Lag (Lag 1, 2, 3)
features_sat = ['VV', 'VH', 'RVI', 'NDVI', 'LSWI', 'EVI', 'BSI']
for lag in [1, 2, 3]: 
    for feat in features_sat:
        df[f'{feat}_lag{lag}'] = df.groupby('segmen')[feat].shift(lag)

df_panel = df.dropna(subset=['Lahan']).reset_index(drop=True)

# ==========================================================
# ⚖️ 3.5. CUSTOM BALANCING DATASET (DINAMIS & PROPORSIONAL)
# ==========================================================
print("\n⚖️ Melakukan penyeimbangan data dinamis...")

df_panel['Lahan'] = df_panel['Lahan'].astype(str).str.strip().str.lower()
df_padi = df_panel[df_panel['Lahan'] == 'padi']
n_padi = len(df_padi)

kelas_non_padi = df_panel[df_panel['Lahan'] != 'padi']['Lahan'].unique()
k_classes = len(kelas_non_padi)

if k_classes > 0:
    print(f"📌 Ditemukan {k_classes} kelas Non-Padi: {kelas_non_padi}")
    
    kuota_per_kelas = n_padi // k_classes
    sisa = n_padi % k_classes
    
    df_non_padi_terpilih = []
    for i, cls in enumerate(kelas_non_padi):
        df_cls = df_panel[df_panel['Lahan'] == cls]
        target_n = kuota_per_kelas + (1 if i < sisa else 0)
        n_ambil = min(target_n, len(df_cls))
        df_non_padi_terpilih.append(df_cls.sample(n=n_ambil, random_state=42))
        
    df_non_padi_balanced = pd.concat(df_non_padi_terpilih)
    df_balanced = pd.concat([df_padi, df_non_padi_balanced])
else:
    df_balanced = df_padi

df_balanced['Label_Biner'] = np.where(df_balanced['Lahan'] == 'padi', 1, 0)

print("\n📊 RINGKASAN DATA SETELAH BALANCING:")
print(f"Total Padi (1)      : {sum(df_balanced['Label_Biner'] == 1)}")
print(f"Total Bukan Padi (0): {sum(df_balanced['Label_Biner'] == 0)}")
print("Rincian kelas asli pada Bukan Padi:")
print(df_balanced[df_balanced['Label_Biner'] == 0]['Lahan'].value_counts())
print("\n")

df_panel = df_balanced

# ----------------------------------------------------------
# SORTING KE-2: Waktu Dulu (KRUSIAL UNTUK TIME SERIES SPLIT)
# ----------------------------------------------------------
df_panel = df_panel.sort_values(['Tahun', 'Bulan', 'segmen']).reset_index(drop=True)

# ==========================================================
# C. Seleksi Semua Fitur
# ==========================================================
spasial = ['Bulan_sin', 'Bulan_cos', 'Tahun'] 
feats_all = spasial + features_sat + [f'{feat}_lag{lag}' for lag in [1, 2, 3] for feat in features_sat]

X = df_panel[feats_all]
y = df_panel['Label_Biner']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# ==========================================================
# 🧠 4. KAMUS KONFIGURASI MODEL & UPGRADED HYPER-TUNING
# ==========================================================
models_config = {
    'Logistic Regression': (LogisticRegression(random_state=42, max_iter=2000), 
                            {'model__C': [0.001, 0.01, 0.1, 1, 10, 100], 
                             'model__solver': ['lbfgs', 'liblinear', 'saga']}),
    
    'CART': (DecisionTreeClassifier(random_state=42), 
             {'model__max_depth': [5, 10, 20, 30, None], 
              'model__min_samples_split': [2, 5, 10, 20],
              'model__min_samples_leaf': [1, 2, 4, 8]}),
    
    'Naive Bayes': (GaussianNB(), 
                    {'model__var_smoothing': np.logspace(-10, -2, 10)}),
    
    'KNN': (KNeighborsClassifier(), 
            {'model__n_neighbors': [3, 5, 7, 9, 11, 15], 
             'model__weights': ['uniform', 'distance'],
             'model__p': [1, 2]}),
    
    'Random Forest': (RandomForestClassifier(random_state=42, n_jobs=-1), 
                      {'model__n_estimators': [100, 200, 300], 
                       'model__max_depth': [10, 20, 30, None],
                       'model__min_samples_split': [2, 5, 10],
                       'model__min_samples_leaf': [1, 2, 4]}),
    
    'XGBoost': (XGBClassifier(random_state=42, tree_method='hist', n_jobs=-1), 
                {'model__n_estimators': [100, 200, 300], 
                 'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
                 'model__max_depth': [3, 5, 7, 9],
                 'model__subsample': [0.8, 0.9, 1.0],
                 'model__colsample_bytree': [0.8, 0.9, 1.0]}),
    
    'LightGBM': (LGBMClassifier(random_state=42, verbose=-1, n_jobs=-1), 
                 {'model__n_estimators': [100, 200, 300], 
                  'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
                  'model__num_leaves': [31, 50, 70, 100],
                  'model__max_depth': [-1, 5, 10, 20],
                  'model__subsample': [0.8, 0.9, 1.0]}),
    
    'CatBoost': (CatBoostClassifier(random_state=42, verbose=0, thread_count=-1), 
                 {'model__iterations': [100, 200, 300], 
                  'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
                  'model__depth': [4, 6, 8, 10],
                  'model__l2_leaf_reg': [1, 3, 5, 7]})
}

if MODEL_PILIHAN not in models_config:
    raise ValueError(f"❌ Model '{MODEL_PILIHAN}' tidak ditemukan! Pilih salah satu dari: {list(models_config.keys())}")

model_base, param_grid = models_config[MODEL_PILIHAN]

pipe = Pipeline([
    ('scaler', StandardScaler()), 
    ('model', model_base)
])

tscv = TimeSeriesSplit(n_splits=3)
# n_iter dinaikkan ke 10 agar ruang pencarian yang sudah diperluas dapat dieksplorasi dengan baik
tuner = RandomizedSearchCV(pipe, param_grid, cv=tscv, scoring='f1_macro', n_iter=10, n_jobs=-1, random_state=42)

print(f"🚀 Memulai Training dan Tuning (n_iter=10) untuk {MODEL_PILIHAN}...")
start_time = time.time()
tuner.fit(X_train, y_train)

# ==========================================================
# 📊 5. EVALUASI & SIMPAN HASIL
# ==========================================================
best_model = tuner.best_estimator_
y_pred = best_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='macro')
kappa = cohen_kappa_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='macro')
rec = recall_score(y_test, y_pred, average='macro')

waktu = time.time() - start_time
print(f"\n✅ {MODEL_PILIHAN} SELESAI!")
print(f"📊 F1-Macro: {f1:.4f} | Acc: {acc:.4f} | Kappa: {kappa:.4f}")
print(f"⏱️ Waktu Training: {waktu:.1f} detik")
print(f"🔧 Parameter Terbaik: {tuner.best_params_}")

# A. Simpan Confusion Matrix
nama_bersih = MODEL_PILIHAN.replace(" ", "")
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix - {MODEL_PILIHAN}')
plt.ylabel('Aktual')
plt.xlabel('Prediksi')
plt.savefig(f'/kaggle/working/CM_{nama_bersih}.png')
plt.close()

# B. Simpan Log ke CSV
file_hasil = f'/kaggle/working/Summary_{nama_bersih}.csv'
log_data = {
    'Model': MODEL_PILIHAN,
    'Accuracy': acc,
    'F1_Macro': f1,
    'Kappa': kappa,
    'Precision_Macro': prec,
    'Recall_Macro': rec,
    'Waktu_Sekon': round(waktu, 2),
    'Best_Params': str(tuner.best_params_)
}

if os.path.exists(file_hasil):
    df_log = pd.read_csv(file_hasil)
    df_log = pd.concat([df_log, pd.DataFrame([log_data])], ignore_index=True)
else:
    df_log = pd.DataFrame([log_data])
df_log.to_csv(file_hasil, index=False)

# C. Simpan Model PKL
joblib.dump(best_model, f'/kaggle/working/Model_{nama_bersih}.pkl')

gc.collect()
print(f"📂 Semua file evaluasi ({MODEL_PILIHAN}) telah disimpan di folder output!")

🔥 MEMULAI NOTEBOOK KHUSUS UNTUK: CART 🔥

🔍 Membaca dataset dari: /kaggle/input/datasets/jejehayu/data-klasifikasi-padi/unik_per_bulan.csv
✅ Dataset berhasil dimuat! Jumlah baris: 151074

⚖️ Melakukan penyeimbangan data dinamis...
📌 Ditemukan 5 kelas Non-Padi: ['bangunan' 'hutan' 'puso' 'tanaman lain' 'air']

📊 RINGKASAN DATA SETELAH BALANCING:
Total Padi (1)      : 50835
Total Bukan Padi (0): 32178
Rincian kelas asli pada Bukan Padi:
Lahan
bangunan        10167
hutan           10167
air             10167
puso              860
tanaman lain      817
Name: count, dtype: int64


🚀 Memulai Training dan Tuning (n_iter=10) untuk CART...

✅ CART SELESAI!
📊 F1-Macro: 0.8990 | Acc: 0.9048 | Kappa: 0.7982
⏱️ Waktu Training: 29.7 detik
🔧 Parameter Terbaik: {'model__min_samples_split': 10, 'model__min_samples_leaf': 1, 'model__max_depth': 10}
📂 Semua file evaluasi (CART) telah disimpan di folder output!
